# 701 — Connectivity Analysis

## Objective

Execute the prospectively frozen Phase 7 perturbational-connectivity analysis using the stable Notebook 700 handoffs.

This notebook performs the first observed program–perturbagen connectivity calculations in Phase 7. It preserves the frozen BING query representation, resistance-like orientation, signature → condition → timepoint → cell-line → lineage → perturbagen hierarchy, conditional assignment-specific null, global Benjamini–Yekutieli family, and prespecified sensitivity/diagnostic boundaries.

## Execution boundary

Notebook 701 must not use Phase 5 or Phase 6 evidence to define, filter, rescue, or reorder primary results.

The primary long-running Monte Carlo cell is explicitly marked **LONG RUN**. It is checkpointed and restartable. Re-running that cell resumes from the highest fully committed checkpoint rather than restarting completed replicates.

The outputs are computational perturbational hypotheses. They do not establish therapeutic reversal, efficacy, causal mechanism, validated targets, or clinical predictiveness.


In [ ]:
# =============================================================================
# Imports and frozen Notebook 701 constants
# =============================================================================

import hashlib
import h5py
import json
import os
import platform
import shutil
import sys
import tempfile
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_info

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
    validate_artifact_registry,
)
from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

RNG_ENTROPY = 701
RNG_NAMESPACES = {
    "primary_conditional_null": 1,
    "competitive_null_sensitivity": 2,
    "outcome_blind_benchmark": 3,
}

PRIMARY_FAMILY_SIZE = 23742
PRIMARY_BY_Q = 0.05
MC_EXCEEDANCE_TARGET = 20
FINAL_BMAX = 5058108
MC_BATCH_SIZE = 20
CHECKPOINT_INTERVAL = 1000

NUMERICAL_RTOL = 1e-10
NUMERICAL_ATOL = 1e-12
SIGNATURE_BLOCK_SIZE = 4096

PRIMARY_MIN_CELL_LINES = 5
PRIMARY_MIN_LINEAGES = 4

print("Notebook 701 constants: FROZEN")
print("Programs:", PROGRAM_IDS)
print("Primary family:", f"{PRIMARY_FAMILY_SIZE:,}")
print("h:", MC_EXCEEDANCE_TARGET)
print("Bmax:", f"{FINAL_BMAX:,}")
print("MC batch size:", MC_BATCH_SIZE)


In [ ]:
# =============================================================================
# Resolve and validate frozen Notebook 700 + LINCS inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase7.700.program_query_definitions",
    "phase7.700.null_stratification_manifest",
    "phase7.700.benchmark_report",
    "phase7.700.analysis_metadata",
    "phase1.110.raw_input_inventory",
    "phase1.110.primary_signature_handoff",
    "phase1.110.primary_perturbagen_coverage",
    "phase1.110.landmark_gene_handoff",
)

missing_input_ids = [
    artifact_id
    for artifact_id in INPUT_ARTIFACT_IDS
    if artifact_id not in artifact_registry["artifacts"]
]

if missing_input_ids:
    raise RuntimeError(
        "Notebook 701 is blocked until Notebook 700 registration passes. "
        "Missing artifacts: "
        + ", ".join(missing_input_ids)
    )

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

program_query_definitions = pd.read_csv(
    input_paths[
        "phase7.700.program_query_definitions"
    ],
    low_memory=False,
)

null_stratification_manifest = pd.read_csv(
    input_paths[
        "phase7.700.null_stratification_manifest"
    ],
    low_memory=False,
)

with input_paths[
    "phase7.700.benchmark_report"
].open("r", encoding="utf-8") as handle:
    benchmark_report = json.load(handle)

with input_paths[
    "phase7.700.analysis_metadata"
].open("r", encoding="utf-8") as handle:
    notebook700_metadata = json.load(handle)

raw_input_inventory = pd.read_csv(
    input_paths[
        "phase1.110.raw_input_inventory"
    ],
    low_memory=False,
)

primary_signature_handoff = pd.read_parquet(
    input_paths[
        "phase1.110.primary_signature_handoff"
    ]
)

primary_perturbagen_coverage = pd.read_csv(
    input_paths[
        "phase1.110.primary_perturbagen_coverage"
    ],
    low_memory=False,
)

landmark_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.landmark_gene_handoff"
    ],
    low_memory=False,
)

if len(program_query_definitions) != 4485:
    raise RuntimeError("Unexpected Notebook 700 query-definition size.")

if len(null_stratification_manifest) != 10174:
    raise RuntimeError("Unexpected Notebook 700 null-manifest size.")

if len(primary_signature_handoff) != 362036:
    raise RuntimeError("Unexpected primary signature universe.")

if len(primary_perturbagen_coverage) != 7914:
    raise RuntimeError("Unexpected primary perturbagen universe.")

if (
    notebook700_metadata["sequential_monte_carlo"]["final_bmax"]
    != FINAL_BMAX
):
    raise RuntimeError("Notebook 700 Bmax does not match Notebook 701.")

if benchmark_report[
    "observed_program_perturbagen_connectivity_inspected"
] is not False:
    raise RuntimeError(
        "Notebook 700 benchmark is not marked outcome-blind."
    )

raw_resource_paths = (
    raw_input_inventory
    .set_index("resource")["relative_path"]
)

LINCS_LEVEL5_TRT_CP_PATH = (
    Paths.root
    / raw_resource_paths.loc["level5_trt_cp"]
)

if not LINCS_LEVEL5_TRT_CP_PATH.is_file():
    raise FileNotFoundError(
        f"Missing LINCS Level 5 GCTX: {LINCS_LEVEL5_TRT_CP_PATH}"
    )

HALLMARK_GMT_PATH = (
    Paths.msigdb
    / "h.all.v2026.1.Hs.symbols.gmt"
)

if not HALLMARK_GMT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing frozen Hallmark GMT: {HALLMARK_GMT_PATH}"
    )

input_hashes = {
    artifact_id: artifact_registry[
        "artifacts"
    ][artifact_id]["sha256"]
    for artifact_id in INPUT_ARTIFACT_IDS
}

print("NOTEBOOK 701 INPUT GATE: PASS")
print("Registered inputs:", len(INPUT_ARTIFACT_IDS))
print("Primary signatures:", f"{len(primary_signature_handoff):,}")
print("Primary perturbagens:", f"{len(primary_perturbagen_coverage):,}")
print("Final Bmax:", f"{FINAL_BMAX:,}")


In [ ]:
# =============================================================================
# Reconstruct frozen query, null, RNG, and competitive matching definitions
# =============================================================================

query_wide = (
    program_query_definitions[
        [
            "gene_id",
            "program_id",
            "frozen_weight",
        ]
    ]
    .pivot(
        index="gene_id",
        columns="program_id",
        values="frozen_weight",
    )
    .reindex(columns=list(PROGRAM_IDS))
    .reset_index()
)

if query_wide[list(PROGRAM_IDS)].isna().any().any():
    raise RuntimeError("Incomplete coupled three-program query weights.")

query_wide = (
    query_wide
    .merge(
        null_stratification_manifest[
            ["gene_id", "stratum_id"]
        ],
        on="gene_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["stratum_id", "gene_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

query_gene_ids = query_wide[
    "gene_id"
].to_numpy(dtype=np.int64)

frozen_weight_matrix = query_wide[
    list(PROGRAM_IDS)
].to_numpy(dtype=np.float64)

frozen_weight_norms = np.linalg.norm(
    frozen_weight_matrix,
    axis=0,
)

query_stratum_ids = query_wide[
    "stratum_id"
].to_numpy(dtype=np.int64)

STRATUM_IDS = tuple(
    sorted(
        np.unique(query_stratum_ids).tolist()
    )
)

query_stratum_row_indices = {
    int(stratum_id): np.flatnonzero(
        query_stratum_ids == int(stratum_id)
    ).astype(np.int64)
    for stratum_id in STRATUM_IDS
}

program_metadata = (
    program_query_definitions[
        [
            "program_id",
            "resistance_like_orientation_multiplier",
        ]
    ]
    .drop_duplicates()
    .set_index("program_id")
    .loc[list(PROGRAM_IDS)]
)

resistance_like_multiplier_vector = (
    program_metadata[
        "resistance_like_orientation_multiplier"
    ]
    .to_numpy(dtype=np.float64)
)

if not np.array_equal(
    resistance_like_multiplier_vector,
    np.array([-1.0, 1.0, 1.0]),
):
    raise RuntimeError(
        "Resistance-like orientation differs from frozen Phase 7 definition."
    )

def make_phase7_rng(
    namespace,
    replicate_id,
    stratum_id,
):
    seed_sequence = np.random.SeedSequence(
        entropy=RNG_ENTROPY,
        spawn_key=(
            int(namespace),
            int(replicate_id),
            int(stratum_id),
        ),
    )

    return np.random.Generator(
        np.random.PCG64DXSM(seed_sequence)
    )


def phase7_uniform_permutation(
    n_positions,
    namespace,
    replicate_id,
    stratum_id,
):
    rng = make_phase7_rng(
        namespace=namespace,
        replicate_id=replicate_id,
        stratum_id=stratum_id,
    )

    return rng.permutation(
        int(n_positions)
    ).astype(
        np.int64,
        copy=False,
    )


def generate_coupled_pseudo_weights(
    replicate_id,
    namespace,
):
    pseudo_weights = frozen_weight_matrix.copy()

    for stratum_id in STRATUM_IDS:
        rows = query_stratum_row_indices[
            int(stratum_id)
        ]

        permutation = phase7_uniform_permutation(
            n_positions=len(rows),
            namespace=namespace,
            replicate_id=replicate_id,
            stratum_id=stratum_id,
        )

        pseudo_weights[
            rows,
            :,
        ] = frozen_weight_matrix[
            rows[permutation],
            :,
        ]

    return pseudo_weights


# -------------------------------------------------------------------------
# Freeze competitive-null matched blocks before observed connectivity.
# -------------------------------------------------------------------------

null_gene_table = (
    null_stratification_manifest
    .sort_values(
        ["stratum_id", "gene_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

real_query_gene_set = set(
    query_gene_ids.tolist()
)

competitive_blocks = []

for stratum_id in STRATUM_IDS:
    stratum_id = int(stratum_id)

    demand_rows = query_stratum_row_indices[
        stratum_id
    ]

    demand = int(len(demand_rows))

    background = (
        null_gene_table.loc[
            null_gene_table["stratum_id"].eq(stratum_id)
            & ~null_gene_table["gene_id"].isin(
                real_query_gene_set
            )
        ]
    )

    if len(background) < demand:
        raise RuntimeError(
            "Competitive-null singleton stratum lacks sufficient "
            "background. A prospectively specified merge implementation "
            "would be required before observed connectivity is inspected."
        )

    competitive_blocks.append(
        {
            "stratum_ids": [stratum_id],
            "rng_stratum_id": stratum_id,
            "query_row_indices": (
                demand_rows.astype(int).tolist()
            ),
            "demand": demand,
            "eligible_background_gene_ids": (
                background["gene_id"]
                .astype(int)
                .sort_values()
                .tolist()
            ),
        }
    )

COMPETITIVE_PLAN_PATH = (
    Paths.perturbational_hypotheses
    / "701_competitive_matching_plan.json"
)

competitive_plan_payload = {
    "status": "FROZEN_BEFORE_OBSERVED_CONNECTIVITY",
    "scientific_results_inspected": False,
    "namespace": RNG_NAMESPACES[
        "competitive_null_sensitivity"
    ],
    "draws": 10000,
    "blocks": competitive_blocks,
}

if COMPETITIVE_PLAN_PATH.exists():
    with COMPETITIVE_PLAN_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        existing_competitive_plan = json.load(handle)

    if existing_competitive_plan != competitive_plan_payload:
        raise RuntimeError(
            "Non-identical competitive matching plan already exists."
        )
else:
    with COMPETITIVE_PLAN_PATH.open(
        "x",
        encoding="utf-8",
    ) as handle:
        json.dump(
            competitive_plan_payload,
            handle,
            indent=2,
            sort_keys=True,
        )
        handle.write("\n")

print("FROZEN QUERY / NULL DEFINITIONS: PASS")
print("Coupled query genes:", f"{len(query_gene_ids):,}")
print("Null strata:", len(STRATUM_IDS))
print("Competitive matched blocks:", len(competitive_blocks))
print("Observed connectivity inspected: NO")


In [ ]:
# =============================================================================
# Reuse or reconstruct the primary BING expression cache
# =============================================================================

primary_signature_metadata = (
    primary_signature_handoff
    .sort_values(
        "gctx_signature_index",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

primary_signature_indices = (
    primary_signature_metadata[
        "gctx_signature_index"
    ]
    .to_numpy(dtype=np.int64)
)

bing_gene_table = (
    null_stratification_manifest[
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
        ]
    ]
    .drop_duplicates("gene_id")
    .sort_values(
        "gene_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

n_bing_genes = len(bing_gene_table)
n_primary_signatures = len(
    primary_signature_metadata
)

TEMP_BING_MATRIX_PATH = (
    Path(tempfile.gettempdir())
    / "phase7_700_primary_bing_matrix.npy"
)

cache_expected_shape = (
    n_bing_genes,
    n_primary_signatures,
)

cache_reused = False

if TEMP_BING_MATRIX_PATH.is_file():
    try:
        candidate_cache = np.load(
            TEMP_BING_MATRIX_PATH,
            mmap_mode="r",
        )

        if (
            candidate_cache.shape
            == cache_expected_shape
            and candidate_cache.dtype
            == np.dtype(np.float32)
        ):
            primary_bing_matrix = (
                candidate_cache
            )
            cache_reused = True
    except Exception:
        cache_reused = False

if not cache_reused:
    projected_bytes = (
        n_bing_genes
        * n_primary_signatures
        * np.dtype(np.float32).itemsize
    )

    free_bytes = shutil.disk_usage(
        TEMP_BING_MATRIX_PATH.parent
    ).free

    if free_bytes < projected_bytes * 1.10:
        raise RuntimeError(
            "Insufficient temporary storage to reconstruct BING cache."
        )

    with h5py.File(
        LINCS_LEVEL5_TRT_CP_PATH,
        mode="r",
    ) as gctx:
        gctx_gene_ids = pd.to_numeric(
            pd.Series(
                [
                    (
                        value.decode("utf-8")
                        if isinstance(value, bytes)
                        else str(value)
                    )
                    for value in gctx[
                        "0/META/ROW/id"
                    ][:]
                ]
            ),
            errors="raise",
        ).astype("int64")

        gctx_gene_lookup = pd.Series(
            np.arange(
                len(gctx_gene_ids),
                dtype=np.int64,
            ),
            index=gctx_gene_ids.to_numpy(),
        )

        bing_gctx_indices = (
            bing_gene_table["gene_id"]
            .map(gctx_gene_lookup)
            .to_numpy(dtype=np.int64)
        )

        primary_bing_matrix = (
            np.lib.format.open_memmap(
                TEMP_BING_MATRIX_PATH,
                mode="w+",
                dtype=np.float32,
                shape=cache_expected_shape,
            )
        )

        matrix = gctx[
            "0/DATA/0/matrix"
        ]

        for block_start in range(
            0,
            matrix.shape[0],
            SIGNATURE_BLOCK_SIZE,
        ):
            block_stop = min(
                block_start
                + SIGNATURE_BLOCK_SIZE,
                matrix.shape[0],
            )

            left = np.searchsorted(
                primary_signature_indices,
                block_start,
                side="left",
            )

            right = np.searchsorted(
                primary_signature_indices,
                block_stop,
                side="left",
            )

            if left == right:
                continue

            block = matrix[
                block_start:block_stop,
                :,
            ]

            local_rows = (
                primary_signature_indices[
                    left:right
                ]
                - block_start
            )

            primary_bing_matrix[
                :,
                left:right,
            ] = block[
                local_rows
            ][
                :,
                bing_gctx_indices
            ].T

        primary_bing_matrix.flush()

    primary_bing_matrix = np.load(
        TEMP_BING_MATRIX_PATH,
        mmap_mode="r",
    )


bing_row_lookup = pd.Series(
    np.arange(
        len(bing_gene_table),
        dtype=np.int64,
    ),
    index=bing_gene_table[
        "gene_id"
    ].to_numpy(dtype=np.int64),
)

query_bing_row_indices = (
    pd.Series(query_gene_ids)
    .map(bing_row_lookup)
)

if query_bing_row_indices.isna().any():
    raise RuntimeError(
        "Frozen query gene absent from BING cache."
    )

query_bing_row_indices = (
    query_bing_row_indices
    .to_numpy(dtype=np.int64)
)

# Independent sentinel validation against raw GCTX.
with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    for bing_row in (
        0,
        len(bing_gene_table) // 2,
        len(bing_gene_table) - 1,
    ):
        gene_id = int(
            bing_gene_table.iloc[
                bing_row
            ]["gene_id"]
        )

        raw_gene_index = int(
            null_stratification_manifest.loc[
                null_stratification_manifest[
                    "gene_id"
                ].eq(gene_id),
                "gctx_gene_index",
            ].iloc[0]
        )

        for signature_position in (
            0,
            len(primary_signature_indices) // 2,
            len(primary_signature_indices) - 1,
        ):
            raw_signature_index = int(
                primary_signature_indices[
                    signature_position
                ]
            )

            cached_value = float(
                primary_bing_matrix[
                    bing_row,
                    signature_position,
                ]
            )

            raw_value = float(
                matrix[
                    raw_signature_index,
                    raw_gene_index,
                ]
            )

            if cached_value != raw_value:
                raise RuntimeError(
                    "BING cache sentinel validation failed."
                )

print("PRIMARY BING CACHE: VALIDATED")
print("Cache action:", "REUSED" if cache_reused else "RECONSTRUCTED")
print("Shape:", primary_bing_matrix.shape)
print("Dtype:", primary_bing_matrix.dtype)
print("Path:", TEMP_BING_MATRIX_PATH)


In [ ]:
# =============================================================================
# Reconstruct and validate the frozen aggregation hierarchy
# =============================================================================

condition_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

timepoint_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_time",
    "pert_time_unit",
]

cell_line_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
]

lineage_keys = [
    "pert_id",
    "cell_lineage",
]


def factorize_group_keys(
    frame,
    columns,
):
    keys = pd.MultiIndex.from_frame(
        frame[columns]
    )

    codes, uniques = pd.factorize(
        keys,
        sort=True,
    )

    return (
        codes.astype(np.int32),
        uniques,
    )


def build_parent_map(
    child_ids,
    parent_ids,
    n_children,
):
    mapping = (
        pd.DataFrame(
            {
                "child_id": child_ids,
                "parent_id": parent_ids,
            }
        )
        .drop_duplicates()
        .sort_values(
            "child_id",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if mapping["child_id"].duplicated().any():
        raise RuntimeError(
            "Hierarchy child maps to multiple parents."
        )

    if not np.array_equal(
        mapping["child_id"].to_numpy(),
        np.arange(
            n_children,
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            "Hierarchy IDs are not complete and contiguous."
        )

    return mapping[
        "parent_id"
    ].to_numpy(dtype=np.int32)


condition_id, condition_index = factorize_group_keys(
    primary_signature_metadata,
    condition_keys,
)

timepoint_id, timepoint_index = factorize_group_keys(
    primary_signature_metadata,
    timepoint_keys,
)

cell_line_id, cell_line_index = factorize_group_keys(
    primary_signature_metadata,
    cell_line_keys,
)

lineage_id, lineage_index = factorize_group_keys(
    primary_signature_metadata,
    lineage_keys,
)

perturbagen_id, perturbagen_index = factorize_group_keys(
    primary_signature_metadata,
    ["pert_id"],
)

condition_to_timepoint = build_parent_map(
    condition_id,
    timepoint_id,
    len(condition_index),
)

timepoint_to_cell_line = build_parent_map(
    timepoint_id,
    cell_line_id,
    len(timepoint_index),
)

cell_line_to_lineage = build_parent_map(
    cell_line_id,
    lineage_id,
    len(cell_line_index),
)

lineage_to_perturbagen = build_parent_map(
    lineage_id,
    perturbagen_id,
    len(lineage_index),
)

condition_to_perturbagen = build_parent_map(
    condition_id,
    perturbagen_id,
    len(condition_index),
)

timepoint_to_perturbagen = build_parent_map(
    timepoint_id,
    perturbagen_id,
    len(timepoint_index),
)

cell_line_to_perturbagen_direct = build_parent_map(
    cell_line_id,
    perturbagen_id,
    len(cell_line_index),
)


def build_median_reduction_plan(
    parent_ids,
):
    parent_ids = np.asarray(
        parent_ids,
        dtype=np.int64,
    )

    order = np.argsort(
        parent_ids,
        kind="stable",
    )

    sorted_parent_ids = parent_ids[
        order
    ]

    unique_parents, starts, counts = np.unique(
        sorted_parent_ids,
        return_index=True,
        return_counts=True,
    )

    if not np.array_equal(
        unique_parents,
        np.arange(
            len(unique_parents),
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            "Parent IDs are not complete and contiguous."
        )

    plan = []

    for group_size in np.unique(
        counts
    ):
        mask = counts == group_size

        plan.append(
            {
                "group_size": int(group_size),
                "parent_ids": unique_parents[
                    mask
                ].astype(np.int32),
                "child_indices": order[
                    starts[mask, np.newaxis]
                    + np.arange(
                        group_size,
                        dtype=np.int64,
                    )
                ].astype(np.int32),
            }
        )

    return plan, len(unique_parents)


def apply_median_reduction(
    values,
    plan,
    n_parents,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    was_1d = values.ndim == 1

    if was_1d:
        values = values[
            :,
            np.newaxis,
        ]

    reduced = np.empty(
        (
            n_parents,
            values.shape[1],
        ),
        dtype=np.float64,
    )

    for block in plan:
        reduced[
            block["parent_ids"],
            :,
        ] = np.median(
            values[
                block["child_indices"],
                :,
            ],
            axis=1,
        )

    return (
        reduced[:, 0]
        if was_1d
        else reduced
    )


aggregation_plans = {
    "signature_to_condition": build_median_reduction_plan(
        condition_id
    ),
    "condition_to_timepoint": build_median_reduction_plan(
        condition_to_timepoint
    ),
    "timepoint_to_cell_line": build_median_reduction_plan(
        timepoint_to_cell_line
    ),
    "cell_line_to_lineage": build_median_reduction_plan(
        cell_line_to_lineage
    ),
    "lineage_to_perturbagen": build_median_reduction_plan(
        lineage_to_perturbagen
    ),
}


def aggregate_signature_scores(
    signature_scores,
):
    values = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    condition = apply_median_reduction(
        values,
        *aggregation_plans[
            "signature_to_condition"
        ],
    )

    timepoint = apply_median_reduction(
        condition,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
    )

    cell_line = apply_median_reduction(
        timepoint,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
    )

    lineage = apply_median_reduction(
        cell_line,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
    )

    perturbagen = apply_median_reduction(
        lineage,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
    )

    return {
        "condition": condition,
        "timepoint": timepoint,
        "cell_line": cell_line,
        "lineage": lineage,
        "perturbagen": perturbagen,
    }


level_to_perturbagen = {
    "condition": condition_to_perturbagen,
    "timepoint": timepoint_to_perturbagen,
    "cell_line": cell_line_to_perturbagen_direct,
    "lineage": lineage_to_perturbagen,
    "perturbagen": np.arange(
        len(perturbagen_index),
        dtype=np.int32,
    ),
}

signature_indices_by_perturbagen = [
    np.flatnonzero(
        perturbagen_id == pert_index
    ).astype(np.int64)
    for pert_index in range(
        len(perturbagen_index)
    )
]

expected_counts = {
    "signatures": 362036,
    "conditions": 325190,
    "timepoints": 113097,
    "cell_lines": 90673,
    "lineages": 55827,
    "perturbagens": 7914,
}

observed_counts = {
    "signatures": len(primary_signature_metadata),
    "conditions": len(condition_index),
    "timepoints": len(timepoint_index),
    "cell_lines": len(cell_line_index),
    "lineages": len(lineage_index),
    "perturbagens": len(perturbagen_index),
}

if observed_counts != expected_counts:
    raise RuntimeError(
        f"Frozen hierarchy mismatch: {observed_counts}"
    )

print("FROZEN HIERARCHY: VALIDATED")
print(observed_counts)


In [ ]:
# =============================================================================
# Define primary scoring, active batching, checkpointing, and multiplicity
# =============================================================================

def score_observed_primary():
    n_signatures = len(
        primary_signature_metadata
    )

    signature_raw = np.empty(
        (
            n_signatures,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    signature_norms = np.empty(
        n_signatures,
        dtype=np.float64,
    )

    for start in range(
        0,
        n_signatures,
        SIGNATURE_BLOCK_SIZE,
    ):
        stop = min(
            start + SIGNATURE_BLOCK_SIZE,
            n_signatures,
        )

        block = np.asarray(
            primary_bing_matrix[
                query_bing_row_indices,
                start:stop,
            ],
            dtype=np.float64,
        )

        norms = np.linalg.norm(
            block,
            axis=0,
        )

        if not np.isfinite(block).all():
            raise RuntimeError(
                "Primary BING block contains non-finite values."
            )

        if not np.all(norms > 0):
            raise RuntimeError(
                "Primary query-support vector has zero norm."
            )

        numerators = (
            frozen_weight_matrix.T
            @ block
        )

        scores = (
            numerators
            / (
                frozen_weight_norms[
                    :,
                    np.newaxis,
                ]
                * norms[
                    np.newaxis,
                    :
                ]
            )
        ).T

        if np.max(
            np.abs(scores)
        ) > 1 + NUMERICAL_ATOL:
            raise RuntimeError(
                "Primary cosine exceeded theoretical range."
            )

        signature_raw[
            start:stop,
            :,
        ] = scores

        signature_norms[
            start:stop
        ] = norms

    return (
        signature_raw,
        signature_norms,
    )


def generate_pseudo_weight_batch(
    replicate_ids,
    namespace,
):
    return np.stack(
        [
            generate_coupled_pseudo_weights(
                replicate_id=int(
                    replicate_id
                ),
                namespace=namespace,
            )
            for replicate_id in (
                replicate_ids
            )
        ],
        axis=0,
    )


def score_signature_block_batch(
    signature_block,
    signature_norms,
    weight_tensor,
):
    signature_block = np.asarray(
        signature_block,
        dtype=np.float64,
    )

    signature_norms = np.asarray(
        signature_norms,
        dtype=np.float64,
    )

    weight_tensor = np.asarray(
        weight_tensor,
        dtype=np.float64,
    )

    n_replicates = (
        weight_tensor.shape[0]
    )

    flat_weights = (
        weight_tensor
        .transpose(1, 0, 2)
        .reshape(
            weight_tensor.shape[1],
            n_replicates
            * len(PROGRAM_IDS),
        )
    )

    numerators = (
        flat_weights.T
        @ signature_block
    )

    repeated_weight_norms = np.tile(
        frozen_weight_norms,
        n_replicates,
    )

    scores = (
        numerators
        / (
            repeated_weight_norms[
                :,
                np.newaxis,
            ]
            * signature_norms[
                np.newaxis,
                :
            ]
        )
    )

    return (
        scores.T
        .reshape(
            signature_block.shape[1],
            n_replicates,
            len(PROGRAM_IDS),
        )
    )


def apply_active_union_reduction(
    values,
    plan,
    n_parents,
    parent_to_perturbagen,
    active_perturbagen_union,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    reduced = np.full(
        (
            n_parents,
            values.shape[1],
        ),
        np.nan,
        dtype=np.float64,
    )

    for block in plan:
        parent_ids = block[
            "parent_ids"
        ].astype(
            np.int64,
            copy=False,
        )

        active_parent = (
            active_perturbagen_union[
                parent_to_perturbagen[
                    parent_ids
                ]
            ]
        )

        if not active_parent.any():
            continue

        selected_parent_ids = (
            parent_ids[
                active_parent
            ]
        )

        selected_child_indices = (
            block[
                "child_indices"
            ][
                active_parent,
                :
            ]
        )

        selected_values = values[
            selected_child_indices,
            :,
        ]

        if not np.isfinite(
            selected_values
        ).all():
            raise RuntimeError(
                "Active hierarchical aggregation encountered "
                "non-finite child values."
            )

        reduced[
            selected_parent_ids,
            :,
        ] = np.median(
            selected_values,
            axis=1,
        )

    return reduced


def aggregate_active_batch_to_perturbagen(
    active_signature_scores,
    active_signature_indices,
    active_perturbagen_union,
):
    active_signature_scores = np.asarray(
        active_signature_scores,
        dtype=np.float64,
    )

    n_columns = (
        active_signature_scores.shape[1]
    )

    values = np.full(
        (
            len(primary_signature_metadata),
            n_columns,
        ),
        np.nan,
        dtype=np.float64,
    )

    values[
        active_signature_indices,
        :,
    ] = active_signature_scores

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "signature_to_condition"
        ],
        parent_to_perturbagen=(
            condition_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
        parent_to_perturbagen=(
            timepoint_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
        parent_to_perturbagen=(
            cell_line_to_perturbagen_direct
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
        parent_to_perturbagen=(
            lineage_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
        parent_to_perturbagen=np.arange(
            len(perturbagen_index),
            dtype=np.int32,
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    return values


CHECKPOINT_SCHEMA_VERSION = (
    "phase7_701_primary_mc_v1"
)

PRIMARY_CHECKPOINT_PATH = (
    Paths.perturbational_hypotheses
    / "701_primary_mc_checkpoint.npz"
)


def atomic_write_primary_checkpoint(
    path,
    next_replicate_id,
    mc_g,
    mc_L,
    active_mask,
    stop_code,
    family_hash,
    observed_hash,
):
    path = Path(path)

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "wb"
    ) as handle:
        np.savez(
            handle,
            schema_version=np.array(
                [CHECKPOINT_SCHEMA_VERSION]
            ),
            final_bmax=np.array(
                [FINAL_BMAX],
                dtype=np.int64,
            ),
            h=np.array(
                [MC_EXCEEDANCE_TARGET],
                dtype=np.int64,
            ),
            next_replicate_id=np.array(
                [next_replicate_id],
                dtype=np.int64,
            ),
            mc_g=np.asarray(
                mc_g,
                dtype=np.int32,
            ),
            mc_L=np.asarray(
                mc_L,
                dtype=np.int32,
            ),
            active_mask=np.asarray(
                active_mask,
                dtype=bool,
            ),
            stop_code=np.asarray(
                stop_code,
                dtype=np.int8,
            ),
            family_hash=np.array(
                [family_hash]
            ),
            observed_hash=np.array(
                [observed_hash]
            ),
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def read_primary_checkpoint(
    path,
    family_hash,
    observed_hash,
):
    with np.load(
        path,
        allow_pickle=False,
    ) as checkpoint:
        if str(
            checkpoint[
                "schema_version"
            ][0]
        ) != CHECKPOINT_SCHEMA_VERSION:
            raise RuntimeError(
                "Unexpected primary checkpoint schema."
            )

        if int(
            checkpoint[
                "final_bmax"
            ][0]
        ) != FINAL_BMAX:
            raise RuntimeError(
                "Checkpoint Bmax mismatch."
            )

        if int(
            checkpoint["h"][0]
        ) != MC_EXCEEDANCE_TARGET:
            raise RuntimeError(
                "Checkpoint h mismatch."
            )

        if str(
            checkpoint[
                "family_hash"
            ][0]
        ) != family_hash:
            raise RuntimeError(
                "Checkpoint family hash mismatch."
            )

        if str(
            checkpoint[
                "observed_hash"
            ][0]
        ) != observed_hash:
            raise RuntimeError(
                "Checkpoint observed-score hash mismatch."
            )

        return {
            "next_replicate_id": int(
                checkpoint[
                    "next_replicate_id"
                ][0]
            ),
            "mc_g": checkpoint[
                "mc_g"
            ].copy(),
            "mc_L": checkpoint[
                "mc_L"
            ].copy(),
            "active_mask": checkpoint[
                "active_mask"
            ].copy(),
            "stop_code": checkpoint[
                "stop_code"
            ].copy(),
        }


def adjust_bh(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=np.float64,
    )

    m = len(p_values)

    order = np.argsort(
        p_values,
        kind="mergesort",
    )

    ranked = p_values[
        order
    ]

    adjusted_ranked = (
        ranked
        * m
        / np.arange(
            1,
            m + 1,
            dtype=np.float64,
        )
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted = np.empty(
        m,
        dtype=np.float64,
    )

    adjusted[
        order
    ] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


def adjust_by(
    p_values,
):
    m = len(p_values)

    harmonic = np.sum(
        1.0
        / np.arange(
            1,
            m + 1,
            dtype=np.float64,
        )
    )

    return np.minimum(
        adjust_bh(p_values)
        * harmonic,
        1.0,
    )


# Independent multiplicity reference on a toy vector.
toy_p = np.array(
    [0.01, 0.04, 0.03, 0.002],
    dtype=np.float64,
)

np.testing.assert_allclose(
    adjust_bh(toy_p),
    np.array(
        [0.02, 0.04, 0.04, 0.008],
        dtype=np.float64,
    ),
    rtol=0,
    atol=1e-15,
)

print("PRIMARY EXECUTION FUNCTIONS: VALIDATED")
print("Checkpoint:", project_relative_path(PRIMARY_CHECKPOINT_PATH))
print("Checkpoint interval:", CHECKPOINT_INTERVAL)


In [ ]:
# =============================================================================
# Compute observed primary connectivity and freeze the primary hypothesis family
# =============================================================================

observed_signature_raw, query_signature_norms = (
    score_observed_primary()
)

observed_raw_aggregation = (
    aggregate_signature_scores(
        observed_signature_raw
    )
)

observed_rl_aggregation = {
    level: (
        values
        * resistance_like_multiplier_vector[
            np.newaxis,
            :
        ]
    )
    for level, values in (
        observed_raw_aggregation.items()
    )
}

perturbagen_ids = np.array(
    sorted(
        primary_signature_metadata[
            "pert_id"
        ].unique()
    ),
    dtype=object,
)

if len(perturbagen_ids) != 7914:
    raise RuntimeError(
        "Unexpected perturbagen family size."
    )

# Factorization with sort=True must reproduce the same perturbagen order.
perturbagen_index_values = np.array(
    perturbagen_index.get_level_values(
        "pert_id"
    ).tolist(),
    dtype=object,
)

if not np.array_equal(
    perturbagen_ids,
    perturbagen_index_values,
):
    raise RuntimeError(
        "Perturbagen index order mismatch."
    )


# -------------------------------------------------------------------------
# Lineage-level stable table.
# -------------------------------------------------------------------------

lineage_index_frame = (
    lineage_index
    .to_frame(index=False)
)

lineage_cell_counts = (
    pd.DataFrame(
        {
            "lineage_id": (
                cell_line_to_lineage
            ),
            "cell_line_id": np.arange(
                len(cell_line_index),
                dtype=np.int64,
            ),
        }
    )
    .groupby(
        "lineage_id",
        as_index=False,
    )
    .agg(
        n_evaluable_cell_lines=(
            "cell_line_id",
            "size",
        )
    )
)

lineage_rows = []

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    block = (
        lineage_index_frame
        .copy()
        .assign(
            program_id=program_id,
            resistance_like_orientation_multiplier=int(
                resistance_like_multiplier_vector[
                    program_index
                ]
            ),
            C_raw=observed_raw_aggregation[
                "lineage"
            ][:, program_index],
            C_RL=observed_rl_aggregation[
                "lineage"
            ][:, program_index],
        )
        .reset_index(
            names="lineage_id"
        )
        .merge(
            lineage_cell_counts,
            on="lineage_id",
            how="left",
            validate="one_to_one",
        )
    )

    lineage_rows.append(block)

lineage_connectivity = (
    pd.concat(
        lineage_rows,
        ignore_index=True,
    )
    .sort_values(
        [
            "program_id",
            "pert_id",
            "cell_lineage",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Coverage/evaluability counts.
# -------------------------------------------------------------------------

signature_counts = (
    primary_signature_metadata
    .groupby(
        "pert_id",
        as_index=False,
    )
    .agg(
        n_evaluable_signatures=(
            "sig_id",
            "size",
        ),
        n_evaluable_cell_lines=(
            "cell_iname",
            "nunique",
        ),
        n_evaluable_lineages=(
            "cell_lineage",
            "nunique",
        ),
    )
)

condition_counts = (
    primary_signature_metadata[
        condition_keys
    ]
    .drop_duplicates()
    .groupby(
        "pert_id",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_evaluable_conditions",
        }
    )
)

timepoint_counts = (
    primary_signature_metadata[
        timepoint_keys
    ]
    .drop_duplicates()
    .groupby(
        "pert_id",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_evaluable_timepoints",
        }
    )
)

coverage_counts = (
    signature_counts
    .merge(
        condition_counts,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        timepoint_counts,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)

lineage_summary = (
    lineage_connectivity
    .groupby(
        [
            "program_id",
            "pert_id",
        ],
        as_index=False,
    )
    .agg(
        lineage_q25_C_RL=(
            "C_RL",
            lambda values: float(
                np.quantile(
                    values.to_numpy(
                        dtype=np.float64
                    ),
                    0.25,
                )
            ),
        ),
        lineage_q75_C_RL=(
            "C_RL",
            lambda values: float(
                np.quantile(
                    values.to_numpy(
                        dtype=np.float64
                    ),
                    0.75,
                )
            ),
        ),
        fraction_negative_lineages=(
            "C_RL",
            lambda values: float(
                np.mean(
                    values.to_numpy(
                        dtype=np.float64
                    ) < 0
                )
            ),
        ),
    )
)

lineage_summary[
    "lineage_iqr_C_RL"
] = (
    lineage_summary[
        "lineage_q75_C_RL"
    ]
    - lineage_summary[
        "lineage_q25_C_RL"
    ]
)


# -------------------------------------------------------------------------
# Complete frozen 23,742 hypothesis family.
# -------------------------------------------------------------------------

primary_family_rows = []

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    block = pd.DataFrame(
        {
            "program_id": program_id,
            "program_index": program_index,
            "pert_id": perturbagen_ids,
            "perturbagen_index": np.arange(
                len(perturbagen_ids),
                dtype=np.int64,
            ),
            "resistance_like_orientation_multiplier": int(
                resistance_like_multiplier_vector[
                    program_index
                ]
            ),
            "C_raw": observed_raw_aggregation[
                "perturbagen"
            ][:, program_index],
            "C_RL": observed_rl_aggregation[
                "perturbagen"
            ][:, program_index],
        }
    )

    primary_family_rows.append(block)

primary_family = (
    pd.concat(
        primary_family_rows,
        ignore_index=True,
    )
    .merge(
        coverage_counts,
        on="pert_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        lineage_summary,
        on=[
            "program_id",
            "pert_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

primary_family[
    "evaluability_status"
] = np.where(
    (
        primary_family[
            "n_evaluable_cell_lines"
        ]
        >= PRIMARY_MIN_CELL_LINES
    )
    & (
        primary_family[
            "n_evaluable_lineages"
        ]
        >= PRIMARY_MIN_LINEAGES
    ),
    "EVALUABLE",
    "NOT_EVALUABLE",
)

primary_family[
    "directional_status"
] = np.where(
    primary_family[
        "evaluability_status"
    ].eq("EVALUABLE")
    & primary_family[
        "C_RL"
    ].lt(0),
    "INVERSE",
    np.where(
        primary_family[
            "evaluability_status"
        ].eq("NOT_EVALUABLE"),
        "NOT_EVALUABLE",
        "NON_INVERSE",
    ),
)

active_primary_mask_2d = (
    primary_family[
        "directional_status"
    ]
    .eq("INVERSE")
    .to_numpy()
    .reshape(
        len(PROGRAM_IDS),
        len(perturbagen_ids),
    )
    .T
)

family_key_text = "\n".join(
    primary_family[
        ["program_id", "pert_id"]
    ]
    .astype(str)
    .agg("\t".join, axis=1)
    .tolist()
)

PRIMARY_FAMILY_HASH = hashlib.sha256(
    family_key_text.encode("utf-8")
).hexdigest()

observed_hash_payload = np.ascontiguousarray(
    primary_family[
        "C_RL"
    ].to_numpy(dtype=np.float64)
).tobytes()

OBSERVED_PRIMARY_HASH = hashlib.sha256(
    observed_hash_payload
).hexdigest()

if len(primary_family) != PRIMARY_FAMILY_SIZE:
    raise RuntimeError(
        "Primary hypothesis family is not 23,742."
    )

if not np.isfinite(
    primary_family[
        ["C_raw", "C_RL"]
    ].to_numpy(dtype=np.float64)
).all():
    raise RuntimeError(
        "Observed primary connectivity contains non-finite values."
    )

print("OBSERVED PRIMARY CONNECTIVITY: COMPUTED")
print("Primary hypotheses:", f"{len(primary_family):,}")
print(
    "Inverse/evaluable hypotheses entering MC:",
    f"{int(active_primary_mask_2d.sum()):,}",
)
print(
    "Non-inverse hypotheses:",
    f"{primary_family['directional_status'].eq('NON_INVERSE').sum():,}",
)
print(
    "Not-evaluable hypotheses:",
    f"{primary_family['evaluability_status'].eq('NOT_EVALUABLE').sum():,}",
)
print("Primary family hash:", PRIMARY_FAMILY_HASH)
print("Observed-score hash:", OBSERVED_PRIMARY_HASH)


## LONG RUN — Primary conditional randomization

The next cell is the long-running primary Phase 7 Monte Carlo analysis.

It uses namespace 1, h = 20, Bmax = 5,058,108, batch size 20, exact replicate-ordered stopping, and an atomic checkpoint every 1,000 completed global replicates.

If execution is interrupted, rerun the notebook through this cell. The cell validates the checkpoint against the frozen family and observed-score hashes and resumes from the next uncommitted replicate.

Do not delete the checkpoint while the analysis is incomplete.


In [ ]:
# =============================================================================
# LONG RUN — Primary conditional randomization with checkpoint/restart
# =============================================================================

observed_rl_matrix = (
    primary_family[
        "C_RL"
    ]
    .to_numpy(dtype=np.float64)
    .reshape(
        len(PROGRAM_IDS),
        len(perturbagen_ids),
    )
    .T
)

if PRIMARY_CHECKPOINT_PATH.exists():
    restored = read_primary_checkpoint(
        PRIMARY_CHECKPOINT_PATH,
        family_hash=PRIMARY_FAMILY_HASH,
        observed_hash=OBSERVED_PRIMARY_HASH,
    )

    next_replicate_id = restored[
        "next_replicate_id"
    ]

    mc_g = restored[
        "mc_g"
    ]

    mc_L = restored[
        "mc_L"
    ]

    active_mask = restored[
        "active_mask"
    ]

    stop_code = restored[
        "stop_code"
    ]

    print(
        "PRIMARY MC: RESUMING FROM CHECKPOINT"
    )
    print(
        "Next replicate:",
        f"{next_replicate_id:,}",
    )
    print(
        "Active hypotheses:",
        f"{active_mask.sum():,}",
    )

else:
    next_replicate_id = 0

    mc_g = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=np.int32,
    )

    mc_L = np.zeros_like(
        mc_g,
        dtype=np.int32,
    )

    active_mask = (
        active_primary_mask_2d.copy()
    )

    # stop_code:
    # 0 = not sampled / non-inverse / not-evaluable
    # 1 = stopped at h exceedances
    # 2 = reached Bmax with g < h
    stop_code = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=np.int8,
    )

    print(
        "PRIMARY MC: STARTING NEW EXECUTION"
    )
    print(
        "Active hypotheses:",
        f"{active_mask.sum():,}",
    )


execution_start = time.perf_counter()

while (
    next_replicate_id < FINAL_BMAX
    and active_mask.any()
):
    batch_start = int(
        next_replicate_id
    )

    batch_stop = min(
        batch_start + MC_BATCH_SIZE,
        FINAL_BMAX,
    )

    replicate_ids = np.arange(
        batch_start,
        batch_stop,
        dtype=np.int64,
    )

    active_perturbagen_union = (
        active_mask.any(axis=1)
    )

    active_perturbagen_ids = np.flatnonzero(
        active_perturbagen_union
    )

    active_signature_indices = np.concatenate(
        [
            signature_indices_by_perturbagen[
                int(pert_index)
            ]
            for pert_index in (
                active_perturbagen_ids
            )
        ]
    ).astype(
        np.int64,
        copy=False,
    )

    weight_tensor = (
        generate_pseudo_weight_batch(
            replicate_ids=replicate_ids,
            namespace=RNG_NAMESPACES[
                "primary_conditional_null"
            ],
        )
    )

    batch_size_actual = len(
        replicate_ids
    )

    active_signature_scores = np.empty(
        (
            len(active_signature_indices),
            batch_size_actual,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    for local_start in range(
        0,
        len(active_signature_indices),
        SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + SIGNATURE_BLOCK_SIZE,
            len(active_signature_indices),
        )

        signature_positions = (
            active_signature_indices[
                local_start:local_stop
            ]
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    query_bing_row_indices,
                    signature_positions,
                )
            ],
            dtype=np.float64,
        )

        active_signature_scores[
            local_start:local_stop,
            :,
            :,
        ] = score_signature_block_batch(
            signature_block=signature_block,
            signature_norms=(
                query_signature_norms[
                    signature_positions
                ]
            ),
            weight_tensor=weight_tensor,
        )

    flat_signature_scores = (
        active_signature_scores
        .reshape(
            len(active_signature_indices),
            batch_size_actual
            * len(PROGRAM_IDS),
        )
    )

    batch_perturbagen_raw = (
        aggregate_active_batch_to_perturbagen(
            active_signature_scores=(
                flat_signature_scores
            ),
            active_signature_indices=(
                active_signature_indices
            ),
            active_perturbagen_union=(
                active_perturbagen_union
            ),
        )
        .reshape(
            len(perturbagen_ids),
            batch_size_actual,
            len(PROGRAM_IDS),
        )
    )

    batch_perturbagen_rl = (
        batch_perturbagen_raw
        * resistance_like_multiplier_vector[
            np.newaxis,
            np.newaxis,
            :,
        ]
    )

    # Sequential bookkeeping remains exact and replicate ordered.
    for batch_offset, replicate_id in enumerate(
        replicate_ids
    ):
        active_before = (
            active_mask.copy()
        )

        if not active_before.any():
            break

        null_scores = (
            batch_perturbagen_rl[
                :,
                batch_offset,
                :,
            ]
        )

        exceedance = (
            null_scores
            <= observed_rl_matrix
        )

        mc_g[
            active_before
        ] += exceedance[
            active_before
        ].astype(np.int32)

        mc_L[
            active_before
        ] = int(
            replicate_id + 1
        )

        newly_stopped = (
            active_before
            & (
                mc_g
                >= MC_EXCEEDANCE_TARGET
            )
        )

        stop_code[
            newly_stopped
        ] = 1

        active_mask[
            newly_stopped
        ] = False

    next_replicate_id = int(
        batch_stop
    )

    if (
        next_replicate_id
        % CHECKPOINT_INTERVAL
        == 0
        or next_replicate_id
        == FINAL_BMAX
        or not active_mask.any()
    ):
        atomic_write_primary_checkpoint(
            path=PRIMARY_CHECKPOINT_PATH,
            next_replicate_id=(
                next_replicate_id
            ),
            mc_g=mc_g,
            mc_L=mc_L,
            active_mask=active_mask,
            stop_code=stop_code,
            family_hash=PRIMARY_FAMILY_HASH,
            observed_hash=OBSERVED_PRIMARY_HASH,
        )

        elapsed_hours = (
            time.perf_counter()
            - execution_start
        ) / 3600

        print(
            f"checkpoint replicate={next_replicate_id:,} "
            f"active={active_mask.sum():,} "
            f"elapsed_this_session={elapsed_hours:.2f} h"
        )


if (
    next_replicate_id >= FINAL_BMAX
    and active_mask.any()
):
    stop_code[
        active_mask
    ] = 2

    active_mask[
        :
    ] = False

    atomic_write_primary_checkpoint(
        path=PRIMARY_CHECKPOINT_PATH,
        next_replicate_id=FINAL_BMAX,
        mc_g=mc_g,
        mc_L=mc_L,
        active_mask=active_mask,
        stop_code=stop_code,
        family_hash=PRIMARY_FAMILY_HASH,
        observed_hash=OBSERVED_PRIMARY_HASH,
    )


if active_mask.any():
    raise RuntimeError(
        "Primary MC exited with active hypotheses."
    )

sampled_mask = (
    active_primary_mask_2d
)

if not np.all(
    mc_L[
        sampled_mask
    ] > 0
):
    raise RuntimeError(
        "At least one inverse/evaluable hypothesis "
        "has no completed Monte Carlo draws."
    )

if np.any(
    mc_g[
        sampled_mask
    ] > MC_EXCEEDANCE_TARGET
):
    raise RuntimeError(
        "Sequential exceedance target was crossed."
    )

print("PRIMARY CONDITIONAL RANDOMIZATION: COMPLETE")
print("Maximum global replicate reached:", f"{next_replicate_id:,}")
print(
    "Stopped at h:",
    f"{(stop_code == 1).sum():,}",
)
print(
    "Reached Bmax with g<h:",
    f"{(stop_code == 2).sum():,}",
)
print(
    "Checkpoint:",
    project_relative_path(PRIMARY_CHECKPOINT_PATH),
)


In [ ]:
# =============================================================================
# Finalize primary conditional p-values, global multiplicity, and stable outputs
# =============================================================================

primary_perturbagen_connectivity = (
    primary_family.copy()
)

p_conditional_matrix = np.ones(
    (
        len(perturbagen_ids),
        len(PROGRAM_IDS),
    ),
    dtype=np.float64,
)

stopped_h = (
    stop_code == 1
)

reached_bmax = (
    stop_code == 2
)

p_conditional_matrix[
    stopped_h
] = (
    MC_EXCEEDANCE_TARGET
    / mc_L[
        stopped_h
    ].astype(np.float64)
)

p_conditional_matrix[
    reached_bmax
] = (
    mc_g[
        reached_bmax
    ].astype(np.float64)
    + 1.0
) / (
    FINAL_BMAX
    + 1.0
)

if np.any(
    p_conditional_matrix <= 0
):
    raise RuntimeError(
        "Zero Monte Carlo p-value encountered."
    )

primary_perturbagen_connectivity[
    "p_conditional"
] = (
    p_conditional_matrix.T.reshape(-1)
)

primary_perturbagen_connectivity[
    "p_for_multiplicity"
] = np.where(
    primary_perturbagen_connectivity[
        "directional_status"
    ].eq("INVERSE"),
    primary_perturbagen_connectivity[
        "p_conditional"
    ],
    1.0,
)

primary_perturbagen_connectivity[
    "mc_L"
] = mc_L.T.reshape(-1)

primary_perturbagen_connectivity[
    "mc_g"
] = mc_g.T.reshape(-1)

stop_reason_lookup = {
    0: "NOT_SAMPLED",
    1: "H_EXCEEDANCES",
    2: "BMAX_G_LT_H",
}

primary_perturbagen_connectivity[
    "mc_stop_reason"
] = [
    stop_reason_lookup[
        int(code)
    ]
    for code in (
        stop_code.T.reshape(-1)
    )
]

primary_perturbagen_connectivity[
    "Bmax"
] = FINAL_BMAX

primary_perturbagen_connectivity[
    "q_BY_global"
] = adjust_by(
    primary_perturbagen_connectivity[
        "p_for_multiplicity"
    ].to_numpy(dtype=np.float64)
)

primary_perturbagen_connectivity[
    "q_BH_global"
] = adjust_bh(
    primary_perturbagen_connectivity[
        "p_for_multiplicity"
    ].to_numpy(dtype=np.float64)
)


# Program-specific multiplicity is descriptive only.
primary_perturbagen_connectivity[
    "q_BY_program_descriptive"
] = np.nan

primary_perturbagen_connectivity[
    "q_BH_program_descriptive"
] = np.nan

for program_id in PROGRAM_IDS:
    mask = (
        primary_perturbagen_connectivity[
            "program_id"
        ].eq(program_id)
    )

    program_p = (
        primary_perturbagen_connectivity.loc[
            mask,
            "p_for_multiplicity",
        ]
        .to_numpy(dtype=np.float64)
    )

    primary_perturbagen_connectivity.loc[
        mask,
        "q_BY_program_descriptive",
    ] = adjust_by(program_p)

    primary_perturbagen_connectivity.loc[
        mask,
        "q_BH_program_descriptive",
    ] = adjust_bh(program_p)


primary_perturbagen_connectivity[
    "primary_support_status"
] = np.select(
    [
        primary_perturbagen_connectivity[
            "evaluability_status"
        ].eq("NOT_EVALUABLE"),
        primary_perturbagen_connectivity[
            "directional_status"
        ].eq("NON_INVERSE"),
        primary_perturbagen_connectivity[
            "q_BY_global"
        ].le(PRIMARY_BY_Q),
    ],
    [
        "NOT_EVALUABLE",
        "NON_INVERSE",
        "PRIMARY_BY_SUPPORTED",
    ],
    default="INVERSE_NOT_BY_SUPPORTED",
)


PHASE7_OUTPUT_DIR = (
    Paths.perturbational_hypotheses
)

PHASE7_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LINEAGE_CONNECTIVITY_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_lineage_connectivity.parquet"
)

PRIMARY_CONNECTIVITY_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_primary_perturbagen_connectivity.csv"
)

lineage_connectivity.to_parquet(
    LINEAGE_CONNECTIVITY_PATH,
    index=False,
)

primary_perturbagen_connectivity.to_csv(
    PRIMARY_CONNECTIVITY_PATH,
    index=False,
)

print("PRIMARY INFERENCE: FINALIZED")
print(
    "PRIMARY_BY_SUPPORTED:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "PRIMARY_BY_SUPPORTED"
        ).sum()
    ),
)
print(
    "INVERSE_NOT_BY_SUPPORTED:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "INVERSE_NOT_BY_SUPPORTED"
        ).sum()
    ),
)
print(
    "NON_INVERSE:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "NON_INVERSE"
        ).sum()
    ),
)
print(
    "NOT_EVALUABLE:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "NOT_EVALUABLE"
        ).sum()
    ),
)
print(
    "Lineage output:",
    project_relative_path(
        LINEAGE_CONNECTIVITY_PATH
    ),
)
print(
    "Primary output:",
    project_relative_path(
        PRIMARY_CONNECTIVITY_PATH
    ),
)
